In [188]:
import torch
import torch.nn as nn
import math
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import sentencepiece as spm

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [189]:
df = pd.read_csv('/content/drive/My Drive/Colab Notebooks/english_to_bangla.csv')
df.head()

,en,bn
0,a child in a pink dress is climbing up a set o...,একটি গোলাপী জামা পরা বাচ্চা মেয়ে একটি বাড়ির প্...
1,a girl going into a wooden building .,একটি মেয়ে শিশু একটি কাঠের বাড়িতে ঢুকছে
2,a little girl climbing into a wooden playhouse .,একটি বাচ্চা তার কাঠের খেলাঘরে উঠছে ।
3,a little girl climbing the stairs to her playh...,ছোট মেয়েটি তার খেলার ঘরের সিড়ি বেয়ে উঠছে
4,a little girl in a pink dress going into a woo...,গোলাপি জামা পড়া ছোট একটি মেয়ে একটি কাঠের তৈরি...


In [191]:
df.tail(10)

,en,bn
39055,a person stands near golden walls .,একজন ব্যক্তি সোনার দেয়ালের কাছে দাঁড়িয়ে আছেন।
39056,a woman behind a scrolled wall is writing,একটি লিখিত দেয়ালের পিছনে একটি মহিলা লিখছেন
39057,a woman standing near a decorated wall writes .,সজ্জিত দেয়ালের কাছে দাঁড়িয়ে এক মহিলা লিখেছেন
39058,the walls are covered in gold and patterns .,দেয়ালগুলি সোনার এবং নিদর্শনগুলিতে আবৃত।
39059,"woman writing on a pad in room with gold , dec...","স্বর্ণ, সজ্জিত দেয়াল সহ ঘরে একটি প্যাডে লিখছে..."
39060,a man in a pink shirt climbs a rock face,গোলাপী শার্টের একটি লোক একটি শিলার উপরে উঠেছিল
39061,a man is rock climbing high in the air .,একজন মানুষ পাথরে চড়ছে অনেক উপরে
39062,a person in a red shirt climbing up a rock fac...,লাল শার্টের একজন ব্যক্তি সহায়তার হাতলগুলিতে ঢ...
39063,a rock climber in a red shirt .,লাল শার্টে একটি রক আরোহী
39064,a rock climber practices on a rock climbing wa...,একটি রক আরোহী একটি শৈল আরোহণ প্রাচীর এর উপর অন...


In [ ]:
def clean_text(text):
    text = str(text)
    text = text.replace("...", "")
    text = text.strip()
    return text

df["en"] = df["en"].apply(clean_text)
df["bn"] = df["bn"].apply(clean_text)


df = df[(df["en"] != "") & (df["bn"] != "")]

In [193]:
with open("en.txt", "w", encoding="utf-8") as f:
    for line in df["en"]:
        f.write(line + "\n")

with open("bn.txt", "w", encoding="utf-8") as f:
    for line in df["bn"]:
        f.write(line + "\n")

In [ ]:


spm.SentencePieceTrainer.train(
    input="en.txt",
    model_prefix="en",
    vocab_size=5689
)

spm.SentencePieceTrainer.train(
    input="bn.txt",
    model_prefix="bn",
    vocab_size=5689
)
sp_en = spm.SentencePieceProcessor(model_file="en.model")
sp_bn = spm.SentencePieceProcessor(model_file="bn.model")
src_vocab_size = sp_en.get_piece_size() 
tgt_vocab_size = sp_bn.get_piece_size()  

In [ ]:
def encode(en, bn):
    src = [1] + sp_en.encode(en) + [2]   
    tgt = [1] + sp_bn.encode(bn) + [2]
    return src, tgt

pairs = [encode(en, bn) for en, bn in zip(df["en"], df["bn"])]

In [ ]:
from sklearn.model_selection import train_test_split


train_pairs, val_pairs = train_test_split(pairs, test_size=0.05, random_state=42)

In [ ]:


class TranslationDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, idx):
        return self.pairs[idx]

def collate_fn(batch):
    src_batch = [torch.tensor(x[0]) for x in batch]
    tgt_batch = [torch.tensor(x[1]) for x in batch]

    src_batch = pad_sequence(src_batch, batch_first=True, padding_value=0)
    tgt_batch = pad_sequence(tgt_batch, batch_first=True, padding_value=0)
    return src_batch, tgt_batch


train_dataset = TranslationDataset(train_pairs)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)


val_dataset = TranslationDataset(val_pairs)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

In [ ]:

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, heads):
        super().__init__()

        self.d_model = d_model
        self.heads = heads
        self.head_dim = d_model // heads

        assert self.head_dim * heads == d_model

        self.q_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)

        self.fc_out = nn.Linear(d_model, d_model)

    def forward(self, Q, K, V, mask=None):
        batch_size = Q.shape[0]

        Q = self.q_linear(Q)
        K = self.k_linear(K)
        V = self.v_linear(V)

        Q = Q.view(batch_size, -1, self.heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, -1, self.heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, -1, self.heads, self.head_dim).transpose(1, 2)

        energy = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)

        if mask is not None:
            energy = energy.masked_fill(mask == 0, float("-1e20"))

        attention = torch.softmax(energy, dim=-1)

        out = torch.matmul(attention, V)

        out = out.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)

        return self.fc_out(out)

In [199]:
class FeedForward(nn.Module):
    def __init__(self, d_model, hidden_dim):
        super().__init__()
        self.fc1 = nn.Linear(d_model, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, d_model)

    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x)))

In [200]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1)

        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)

        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)].to(x.device)

In [201]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, heads, ff_dim, dropout=0.1):
        super().__init__()

        self.attn = MultiHeadAttention(d_model, heads)
        self.ff = FeedForward(d_model, ff_dim)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        attn_out = self.attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_out))

        ff_out = self.ff(x)
        x = self.norm2(x + self.dropout(ff_out))

        return x

In [202]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, heads, ff_dim, dropout=0.1):
        super().__init__()

        self.self_attn = MultiHeadAttention(d_model, heads)
        self.cross_attn = MultiHeadAttention(d_model, heads)

        self.ff = FeedForward(d_model, ff_dim)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, src_mask, tgt_mask):
        _x = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(_x))

        _x = self.cross_attn(x, enc_out, enc_out, src_mask)
        x = self.norm2(x + self.dropout(_x))

        _x = self.ff(x)
        x = self.norm3(x + self.dropout(_x))

        return x

In [203]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, layers, heads, ff_dim):
        super().__init__()

        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos = PositionalEncoding(d_model)

        self.layers = nn.ModuleList(
            [EncoderLayer(d_model, heads, ff_dim) for _ in range(layers)]
        )

    def forward(self, x, mask):
        x = self.pos(self.embed(x))

        for layer in self.layers:
            x = layer(x, mask)

        return x

In [204]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, layers, heads, ff_dim):
        super().__init__()

        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos = PositionalEncoding(d_model)

        self.layers = nn.ModuleList(
            [DecoderLayer(d_model, heads, ff_dim) for _ in range(layers)]
        )

        self.fc = nn.Linear(d_model, vocab_size)

    def forward(self, x, enc_out, src_mask, tgt_mask):
        x = self.pos(self.embed(x))

        for layer in self.layers:
            x = layer(x, enc_out, src_mask, tgt_mask)

        return self.fc(x)

In [205]:

class Transformer(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, d_model=256, layers=3, heads=8, ff_dim=512):
        super().__init__()

        self.encoder = Encoder(src_vocab, d_model, layers, heads, ff_dim)
        self.decoder = Decoder(tgt_vocab, d_model, layers, heads, ff_dim)

    def make_src_mask(self, src):
        # (batch, 1, 1, src_len)
        return (src != 0).unsqueeze(1).unsqueeze(2)

    def make_tgt_mask(self, tgt):
        batch_size, tgt_len = tgt.shape

        # Padding mask → (batch, 1, 1, tgt_len)
        pad_mask = (tgt != 0).unsqueeze(1).unsqueeze(2)

        # Causal mask → (1, 1, tgt_len, tgt_len)
        causal_mask = torch.tril(
            torch.ones((tgt_len, tgt_len), device=tgt.device)
        ).bool().unsqueeze(0).unsqueeze(1)

        # Final mask → (batch, 1, tgt_len, tgt_len)
        return pad_mask & causal_mask

    def forward(self, src, tgt):
        src_mask = self.make_src_mask(src)
        tgt_mask = self.make_tgt_mask(tgt)

        enc_out = self.encoder(src, src_mask)
        out = self.decoder(tgt, enc_out, src_mask, tgt_mask)

        return out

In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


model = Transformer(
    src_vocab=src_vocab_size, 
    tgt_vocab=tgt_vocab_size,  
    d_model=256,
    layers=3,
    heads=8,
    ff_dim=512
).to(device)


criterion = nn.CrossEntropyLoss(ignore_index=0)  
optimizer = optim.Adam(model.parameters(), lr=1e-4)

In [207]:

def token_accuracy(pred, target, pad_idx=0):
    """
    Compute token-level accuracy ignoring padding.
    
    pred: (batch, seq_len, vocab_size)  -> model output logits
    target: (batch, seq_len)           -> target token IDs
    pad_idx: int                        -> padding token ID (usually 0)
    """
    pred_ids = pred.argmax(dim=-1)            # get predicted token indices
    mask = (target != pad_idx)                # ignore padding tokens
    correct = (pred_ids == target) & mask     # only count non-padding correct tokens
    return correct.sum().float() / mask.sum().float()

In [208]:
num_epochs = 5

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    total_acc = 0

    for src, tgt in train_loader:
        src = src.to(device)
        tgt = tgt.to(device)

        tgt_input = tgt[:, :-1]
        tgt_output = tgt[:, 1:]

        output = model(src, tgt_input)

        loss = criterion(output.reshape(-1, output.shape[-1]), tgt_output.reshape(-1))

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        # Token-level accuracy
        total_acc += token_accuracy(output, tgt_output).item()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    avg_acc = total_acc / len(train_loader)


In [209]:
# Validation
model.eval()
val_acc = 0
with torch.no_grad():
        for src, tgt in val_loader:
            src = src.to(device)
            tgt = tgt.to(device)
            tgt_input = tgt[:, :-1]
            tgt_output = tgt[:, 1:]
            output = model(src, tgt_input)
            val_acc += token_accuracy(output, tgt_output).item()
val_acc /= len(val_loader)
print(f"Epoch {epoch+1}/{num_epochs} | "
          f"Train Loss: {avg_loss:.4f} | Train Acc: {avg_acc:.4f} | Val Acc: {val_acc:.4f}")

Epoch 5/5 | Train Loss: 2.8014 | Train Acc: 0.4580 | Val Acc: 0.4433
